In [ ]:
import json
import re
import os
import pandas as pd
import openpyxl
import io
import tokenize
from typing import List, Dict, Optional

# --- Configuration Settings ------------------
examid = 2 # Examination Number

# Use the Excel export file of course grades for name-to-ID matching
grading_excel_path = 'Grades.xlsx' 

# Download student submissions via: Moodle > Assignment > Submissions > Action > Download all submissions.
# Place all downloaded files inside the target_directory.
# Only .py or .ipynb files will be read.
target_directory = './submissions' 

output_excel_name = 'extracted_code_with_userid.xlsx'
# ----------------------------------------------

def _robustly_clean_code(full_code: str) -> str:
    """
    Cleans the code by removing comments and magic commands.
    """
    if not full_code:
        return ""
    # Remove Jupyter magic commands (starting with ! or %)
    full_code = re.sub(r'^[ \t]*(?:!|%).*\n?', '', full_code, flags=re.MULTILINE)
    # Remove multiline strings/docstrings
    code_without_multiline_comments = re.sub(r'[uU]?[rR]?("""|\'\'\').*?\1', '', full_code, flags=re.DOTALL)    
    code_stream = io.StringIO(code_without_multiline_comments)
    output_tokens = []
    try:
        # Tokenize to remove single-line comments and empty lines
        for token in tokenize.generate_tokens(code_stream.readline):
            if token.type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.ENCODING, tokenize.ENDMARKER):
                output_tokens.append(token)
    except Exception:
        return code_without_multiline_comments

    # Rebuild the cleaned code
    rebuilt_code = []
    last_row, last_col = 1, 0
    for token in output_tokens:
        start_row, start_col = token.start
        if start_row > last_row:
            rebuilt_code.append('\\n' * (start_row - last_row) + ' ' * start_col)
        elif start_col > last_col:
            rebuilt_code.append(' ' * (start_col - last_col))
        rebuilt_code.append(token.string)
        last_row, last_col = token.end
    return "\\n".join([line.rstrip() for line in "".join(rebuilt_code).split('\\n') if line.strip()])

def extract_code_from_ipynb(filepath: str) -> Optional[str]:
    """Extracts and cleans code from Jupyter Notebook cells."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
    except Exception: return None
    all_code_parts = [_robustly_clean_code("".join(c.get('source', []))) 
                      for c in notebook.get('cells', []) if c.get('cell_type') == 'code']
    return "\\n".join(filter(None, all_code_parts))

def extract_code_from_py(filepath: str) -> Optional[str]:
    """Extracts and cleans code from a Python file."""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return _robustly_clean_code(f.read())
    except Exception: return None

# Create a mapping of names to ID numbers from the grading Excel file
def get_name_to_id_map(excel_path: str) -> Dict[str, str]:
    df_grading = pd.read_excel(excel_path)
    # Create map in "Lastname Firstname" format
    # A half-width space is used between Lastname and Firstname to match Moodle filenames
    mapping = {}
    for _, row in df_grading.iterrows():
        full_name = f"{row['First name']} {row['Last name']}"
        mapping[full_name] = str(row['ID number'])
    return mapping

def process_notebooks_and_save_excel(directory: str, output_filename: str):
    data_list = []
    
    # 1. Retrieve the student ID map
    name_map = get_name_to_id_map(grading_excel_path)
    
    if not os.path.exists(directory):
        print(f"Error: Directory '{directory}' not found.")
        return

    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        extracted_code = None

        if filename.endswith('.ipynb'):
            extracted_code = extract_code_from_ipynb(filepath)
        elif filename.endswith('.py'):
            extracted_code = extract_code_from_py(filepath)
        
        if extracted_code is not None:
            # 2. Extract "Full Name" from filename (before the first underscore)
            # Example: "Yamanaka Yudai_12345_..." -> "Yamanaka Yudai"
            full_name_in_file = filename.split('_')[0]
            
            # 3. Get ID number from map. If not found, keep the original name.
            user_id = name_map.get(full_name_in_file, f"Unknown({full_name_in_file})")

            data_list.append({
                'filename': filename,
                'username': user_id, # Stores the ID number here
                'extractedcode': extracted_code,
                'examid': examid
            })

    if not data_list:
        print("No target files were found.")
        return

    # Sort data by username (ID number)
    df = pd.DataFrame(data_list)
    df_sorted = df.sort_values(by='username')

    try:
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            df_sorted.to_excel(writer, sheet_name='Sheet1', index=False)
            ws = writer.sheets['Sheet1']
            
            # Formatting: Adjust column widths
            ws.column_dimensions['A'].width = 30
            ws.column_dimensions['B'].width = 15
            ws.column_dimensions['C'].width = 60
            ws.column_dimensions['D'].width = 10
            
            from openpyxl.styles import Alignment
            wrap = Alignment(wrap_text=True, vertical='top')
            center = Alignment(horizontal='center', vertical='top')
            
            # Formatting: Set row height and alignment
            for row in range(1, len(df_sorted) + 2):
                ws.row_dimensions[row].height = 113.4 if row > 1 else 15
                for col in [1, 2, 3]: ws.cell(row, col).alignment = wrap
                ws.cell(row, 4).alignment = center
            
            print(f"✅ Processing Complete: {output_filename} (Items: {len(data_list)})")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    process_notebooks_and_save_excel(target_directory, output_excel_name)